<a href="https://colab.research.google.com/github/KamiSir/FlyRank-internship-tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KamiSir/FlyRank-internship-tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We will build the feature vector by selecting our historical performance metrics, handling missing data (e.g., filling empty search volumes with 0 and missing competition scores with the median), and converting categorical text like content_type into numerical dummy variables. Finally, we will create our target label is_decaying based on the 15% drop rule.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load data
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# 1. Target Label creation (is_decaying)
df['is_decaying'] = (df['trend_pct'] <= -15.0).astype(int)

# 2. Fill missing values
df['search_volume'] = df['search_volume'].fillna(0)
df['competition'] = df['competition'].fillna(df['competition'].median())

# 3. Categorical encoding (dummy variables) for content_type and competition_level
features_df = pd.get_dummies(df, columns=['content_type', 'competition_level'], drop_first=True)

# 4. Select the final feature vector columns
feature_cols = [
    'search_volume', 'competition', 'word_count',
    'ctr', 'avg_position', 'engagement_rate'
] + [col for col in features_df.columns if col.startswith('content_type_') or col.startswith('competition_level_')]

X = features_df[feature_cols]
y = features_df['is_decaying']

print(f"Feature vector shape: {X.shape}")
print(f"Sample Features: {list(X.columns)[:5]}...")

Feature vector shape: (30000, 10)
Sample Features: ['search_volume', 'competition', 'word_count', 'ctr', 'avg_position']...


## 2. Feature notes (meaning, missing, categorical, available-when?)

search_volume: Average monthly searches. Missing values filled with 0 (assuming the tool found no search volume). Exists before prediction.

competition: Keyword difficulty score. Missing values filled with the dataset median. Exists before prediction.

word_count: The length of the article. Exists before prediction.

ctr & avg_position: Historical search performance metrics. Exists before prediction as a rolling average.

content_type & competition_level: Categorical labels of the page template and difficulty. Handled via one-hot encoding (creating separate True/False columns). Exists before prediction.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt
We must check for target leakage, specifically making sure we aren't including metrics that are derived from the future or directly calculated from the label. I will test the correlation of all numerical columns with our target to find suspiciously high correlations (which usually indicate leakage).


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check correlation of numerical columns with our target
corr_matrix = df.select_dtypes(include=[np.number]).corr()
leakage_check = corr_matrix[['is_decaying']].sort_values(by='is_decaying', ascending=False)

print("--- Highest Correlations with 'is_decaying' ---")
print(leakage_check.head(7))
print("\n--- Lowest Correlations with 'is_decaying' ---")
print(leakage_check.tail(4))

# Notice that 'trend_pct' perfectly predicts the label (-0.61 correlation)
# because the label is directly derived from it.
# We successfully excluded 'trend_pct' from our feature vector (X) in Step 1.

--- Highest Correlations with 'is_decaying' ---
                        is_decaying
is_decaying                1.000000
days_with_impressions      0.232741
word_count                 0.111306
days_since_last_update     0.095562
char_count                 0.090815
impressions_prev_30d       0.024161
scroll_events_90d          0.022531

--- Lowest Correlations with 'is_decaying' ---
                      is_decaying
impressions_last_30d    -0.076558
age_tier_order          -0.136711
content_age_days        -0.145284
trend_pct               -0.146743


## 4. What I excluded and why

content_id & client_id: Excluded because they are unique identifiers, not predictive signals. Including them could cause the model to memorize specific pages rather than learn broad patterns.

trend_pct & trend_direction: Excluded because they are the exact fields used to calculate our target label is_decaying. Including them in training is textbook target leakage.

clicks_90d & pageviews_90d: Excluded because if we are predicting future performance, including post-window performance metrics would be future data leakage.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.